In [ ]:
import cv2
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision


VIDEO_PATH = "IMG_6483.mov"  # Change to your video file
MODEL_PATH = "pose_landmarker_lite.task"
OUTPUT_PATH = "pose_output_video.mp4"  # Output video file
answer = []
# Configuration options
PROCESS_EVERY_N_FRAMES = 3  # Process frames 0, 3, 6, 9, 12, 15... (set to 1 for all frames)
PROCESS_FIRST_N_SECONDS = 10  # Only process first 60 seconds (set to None for full video)
SKIP_UNPROCESSED_FRAMES = True  # If True, unprocessed frames are copied without pose detection

POSE_CONNECTIONS = [
    #Upper
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),

    #Torso
    (11, 23), (12, 24),
    (23, 24),

    # Legs
    (23, 25), (25, 27),
    (24, 26), (26, 28),
]

def draw_pose_landmarks(image, landmarks):
    h, w, _ = image.shape

    for start_idx, end_idx in POSE_CONNECTIONS:
        start = landmarks[start_idx]
        end = landmarks[end_idx]

        x1, y1 = int(start.x * w), int(start.y * h)
        x2, y2 = int(end.x * w), int(end.y * h)

        cv2.line(image, (x1, y1), (x2, y2), (0, 255, 0), 2)

# Open video file
cap = cv2.VideoCapture(VIDEO_PATH)

if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {VIDEO_PATH}")

# Get video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
video_duration = total_frames / fps

print(f"Video info:")
print(f"  - FPS: {fps}")
print(f"  - Resolution: {width}x{height}")
print(f"  - Total frames: {total_frames}")
print(f"  - Duration: {video_duration:.2f} seconds")

# Calculate frame limit based on time constraint
if PROCESS_FIRST_N_SECONDS:
    max_frame = min(total_frames, PROCESS_FIRST_N_SECONDS * fps)
    print(f"  - Processing first {PROCESS_FIRST_N_SECONDS} seconds only (frame 0 to {max_frame})")
else:
    max_frame = total_frames
    print(f"  - Processing full video")

# Calculate expected processed frames
expected_processed = (max_frame + PROCESS_EVERY_N_FRAMES - 1) // PROCESS_EVERY_N_FRAMES
print(f"  - Processing every {PROCESS_EVERY_N_FRAMES} frames (~{expected_processed} frames will be analyzed)")

# Initialize video writer
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_PATH, fourcc, fps, (width, height))

# Initialize MediaPipe detector
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)

options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_poses=1,
)

frame_count = 0
processed_frames = 0
skipped_frames = 0

print("\nStarting processing...")

with vision.PoseLandmarker.create_from_options(options) as detector:
    while True:
        ret, frame = cap.read()
        if not ret or frame_count >= max_frame:
            break  # End of video or reached time limit

        # Determine if we should process this frame
        should_process = (frame_count % PROCESS_EVERY_N_FRAMES == 0)
        
        if should_process:
            # Process frame with pose detection
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

            # Detect pose with timestamp (in milliseconds)
            timestamp_ms = int(frame_count * (1000 / fps))
            result = detector.detect_for_video(mp_image, timestamp_ms)

            # Draw landmarks if detected
            if result.pose_landmarks:
                answer.append(result.pose_landmarks[0])
                draw_pose_landmarks(frame, result.pose_landmarks[0])
            
            processed_frames += 1
            
            # Progress indicator
            if processed_frames % 30 == 0:
                print(f"  Processed {processed_frames} frames (frame #{frame_count})")
        else:
            # Skip processing this frame
            skipped_frames += 1
            if SKIP_UNPROCESSED_FRAMES:
                # Just copy the frame as-is without modifications
                pass  # frame already contains the original image
        
        # Write frame to output video (always write, whether processed or not)
        out.write(frame)
        
        frame_count += 1

# Clean up
cap.release()
out.release()

print(f"\n✓ Processing complete!")
print(f"  - Total frames in output: {frame_count}")
print(f"  - Frames with pose detection: {processed_frames}")
print(f"  - Frames skipped (no detection): {skipped_frames}")
print(f"  - Output video saved to: {OUTPUT_PATH}")

In [ ]:
landmarks = answer.pose_landmarks[0]

# Format: frame n, p11_x, p11_y, p11_z, p12_x, p12_y, p12_z, ...
output_str = f"frame {frame_count}"
for idx in [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]:
    lm = landmarks[idx]
    output_str += f", p{idx}: ({lm.x:.3f}, {lm.y:.3f}, {lm.z:.3f})"
print(output_str)
